# XML 법령 → Qdrant 테스트

`data/ai_basic_law/ai_basic_law.xml`을 읽어 조문을 임베딩하고 Qdrant에 저장합니다.

In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET

from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client
from qdrant_client.models import Distance, PointStruct, VectorParams

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'ai_basic_law.xml'
if not XML_PATH.exists():
    raise FileNotFoundError(f'XML 파일을 찾을 수 없습니다: {XML_PATH}')

detail_xml = ET.parse(XML_PATH).getroot()
print(f'XML 루트: {detail_xml.tag}')

XML 루트: 법령


In [4]:
documents = []
CONTENT_TAGS = ('조문내용', '항내용', '호내용', '목내용')

for unit in detail_xml.findall('.//조문단위'):
    number = unit.findtext('조문번호', '').strip()
    title = unit.findtext('조문제목', '').strip()
    parts = []
    for tag in CONTENT_TAGS:
        for element in unit.findall(f'./{tag}'):
            text = ' '.join(''.join(element.itertext()).split())
            if text and text not in parts:
                parts.append(text)
    content = '\n'.join(parts)
    if content:
        documents.append({
            'id': len(documents) + 1,
            'article': number,
            'title': title,
            'content': content,
            'source': XML_PATH.name,
        })

print(f'Qdrant 저장 대상: {len(documents)}개')
print(documents[0] if documents else '조문을 찾지 못했습니다.')

Qdrant 저장 대상: 54개
{'id': 1, 'article': '1', 'title': '', 'content': '제1장 총칙', 'source': 'ai_basic_law.xml'}


In [5]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_xml'
texts = [doc['content'] for doc in documents]
vectors = await embedding_client.aembed_documents(texts)
VECTOR_SIZE = len(vectors[0])

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

points = [
    PointStruct(id=doc['id'], vector=vector, payload=doc)
    for doc, vector in zip(documents, vectors)
]
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f'{len(points)}개 문서를 Qdrant에 저장했습니다.')

54개 문서를 Qdrant에 저장했습니다.


In [6]:
query = '고영향 인공지능이란 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)
result = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=3)
for point in result.points:
    print(f"[{point.payload['article']} {point.payload['title']}] score={point.score:.4f}")
    print(point.payload['content'][:500], '\n')

[35 고영향 인공지능 영향평가] score=0.5530
제35조(고영향 인공지능 영향평가) 

[33 고영향 인공지능의 확인] score=0.5469
제33조(고영향 인공지능의 확인) 

[13 ] score=0.5461
제1절 인공지능산업 기반 조성 

